In [ ]:
%%sql -r CreateRole
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS LOGISTICS_HOL_ADMIN
  COMMENT = 'MCP HOL - logistics lab role';

GRANT ROLE LOGISTICS_HOL_ADMIN TO ROLE ACCOUNTADMIN;
SET my_user = CURRENT_USER();
GRANT ROLE LOGISTICS_HOL_ADMIN TO USER IDENTIFIER($my_user);

SELECT 'Role LOGISTICS_HOL_ADMIN created' AS STATUS;

In [ ]:
%%sql -r CreateDB
CREATE DATABASE IF NOT EXISTS LOGISTICS_HOL COMMENT = 'MCP HOL - Logistics AI Agent';
CREATE SCHEMA IF NOT EXISTS LOGISTICS_HOL.ANALYTICS COMMENT = 'Gold-layer logistics tables';
CREATE SCHEMA IF NOT EXISTS LOGISTICS_HOL.AGENTS COMMENT = 'Cortex AI objects';

GRANT OWNERSHIP ON DATABASE LOGISTICS_HOL TO ROLE LOGISTICS_HOL_ADMIN COPY CURRENT GRANTS;
GRANT OWNERSHIP ON ALL SCHEMAS IN DATABASE LOGISTICS_HOL TO ROLE LOGISTICS_HOL_ADMIN COPY CURRENT GRANTS;

SELECT 'Database + schemas created' AS STATUS;

In [ ]:
%%sql -r CreateWH
CREATE WAREHOUSE IF NOT EXISTS HOL_WH
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE
  COMMENT = 'MCP HOL compute';

GRANT USAGE ON WAREHOUSE HOL_WH TO ROLE LOGISTICS_HOL_ADMIN;
GRANT OPERATE ON WAREHOUSE HOL_WH TO ROLE LOGISTICS_HOL_ADMIN;

SELECT 'HOL_WH created' AS STATUS;

In [ ]:
%%sql -r GrantPrivs
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE LOGISTICS_HOL_ADMIN;
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

GRANT CREATE TABLE ON SCHEMA LOGISTICS_HOL.ANALYTICS TO ROLE LOGISTICS_HOL_ADMIN;
GRANT CREATE VIEW ON SCHEMA LOGISTICS_HOL.ANALYTICS TO ROLE LOGISTICS_HOL_ADMIN;

GRANT CREATE SEMANTIC VIEW ON SCHEMA LOGISTICS_HOL.AGENTS TO ROLE LOGISTICS_HOL_ADMIN;
GRANT CREATE CORTEX SEARCH SERVICE ON SCHEMA LOGISTICS_HOL.AGENTS TO ROLE LOGISTICS_HOL_ADMIN;
GRANT CREATE AGENT ON SCHEMA LOGISTICS_HOL.AGENTS TO ROLE LOGISTICS_HOL_ADMIN;
GRANT CREATE MCP SERVER ON SCHEMA LOGISTICS_HOL.AGENTS TO ROLE LOGISTICS_HOL_ADMIN;
GRANT CREATE INTEGRATION ON ACCOUNT TO ROLE LOGISTICS_HOL_ADMIN;

SELECT 'Privileges granted' AS STATUS;

In [ ]:
%%sql -r SetContext
USE ROLE LOGISTICS_HOL_ADMIN;
USE DATABASE LOGISTICS_HOL;
USE SCHEMA ANALYTICS;
USE WAREHOUSE HOL_WH;

In [ ]:
%%sql -r CreateShipments
CREATE OR REPLACE TABLE LOGISTICS_HOL.ANALYTICS.SHIPMENTS (
  SHIPMENT_ID VARCHAR(20) NOT NULL,
  CARRIER VARCHAR(30) NOT NULL,
  ORIGIN_CITY VARCHAR(50), ORIGIN_STATE VARCHAR(2),
  DESTINATION_CITY VARCHAR(50), DESTINATION_STATE VARCHAR(2),
  SERVICE_TYPE VARCHAR(30),
  SCHEDULED_DELIVERY DATE, ACTUAL_DELIVERY DATE,
  STATUS VARCHAR(20), IS_LATE BOOLEAN DEFAULT FALSE, DELAY_DAYS NUMBER(3,0) DEFAULT 0,
  WEIGHT_KG NUMBER(8,2), COST_USD NUMBER(8,2),
  CUSTOMER_SEGMENT VARCHAR(20), REGION VARCHAR(20), QUARTER VARCHAR(10),
  CONSTRAINT pk_shipments PRIMARY KEY (SHIPMENT_ID)
);
SELECT 'SHIPMENTS created' AS STATUS;

In [ ]:
%%sql -r CreateIncidents
CREATE OR REPLACE TABLE LOGISTICS_HOL.ANALYTICS.DELIVERY_INCIDENTS (
  INCIDENT_ID VARCHAR(20) NOT NULL,
  SHIPMENT_ID VARCHAR(20), CARRIER VARCHAR(30),
  INCIDENT_DATE DATE, INCIDENT_TYPE VARCHAR(40), REGION VARCHAR(20),
  REPORT_TEXT VARCHAR(2000),
  CONSTRAINT pk_incidents PRIMARY KEY (INCIDENT_ID)
);
SELECT 'DELIVERY_INCIDENTS created' AS STATUS;

In [ ]:
%%sql -r LoadShipments
INSERT INTO LOGISTICS_HOL.ANALYTICS.SHIPMENTS
  (SHIPMENT_ID, CARRIER, ORIGIN_CITY, ORIGIN_STATE, DESTINATION_CITY, DESTINATION_STATE,
   SERVICE_TYPE, SCHEDULED_DELIVERY, ACTUAL_DELIVERY, STATUS, IS_LATE, DELAY_DAYS,
   WEIGHT_KG, COST_USD, CUSTOMER_SEGMENT, REGION, QUARTER)
VALUES
('SHIP-001','FedEx','New York','NY','Boston','MA','Express','2026-01-05','2026-01-07','Delivered',FALSE,0,3.2,48.00,'Enterprise','Northeast','Q1-2026'),
('SHIP-002','FedEx','Chicago','IL','Detroit','MI','Standard Ground','2026-01-10','2026-01-14','Delivered',FALSE,0,12.5,32.00,'Retail','Midwest','Q1-2026'),
('SHIP-003','FedEx','Los Angeles','CA','San Diego','CA','Overnight','2026-01-15','2026-01-16','Delivered',FALSE,0,2.1,89.00,'Healthcare','West','Q1-2026'),
('SHIP-004','FedEx','Houston','TX','Dallas','TX','Standard Ground','2026-01-20','2026-01-24','Delivered',FALSE,0,8.7,28.00,'SMB','Southwest','Q1-2026'),
('SHIP-005','FedEx','Atlanta','GA','Charlotte','NC','Express','2026-01-28','2026-01-30','Delivered',FALSE,0,5.4,55.00,'Retail','Southeast','Q1-2026'),
('SHIP-006','FedEx','Seattle','WA','Portland','OR','Standard Ground','2026-02-03','2026-02-07','Delivered',FALSE,0,14.0,30.00,'SMB','West','Q1-2026'),
('SHIP-007','FedEx','Miami','FL','Tampa','FL','Express','2026-02-10','2026-02-12','Delivered',FALSE,0,4.3,51.00,'Healthcare','Southeast','Q1-2026'),
('SHIP-008','FedEx','Denver','CO','Salt Lake City','UT','Standard Ground','2026-02-18','2026-02-22','Delivered',FALSE,0,9.8,33.00,'Enterprise','West','Q1-2026'),
('SHIP-009','FedEx','Phoenix','AZ','Tucson','AZ','Overnight','2026-03-02','2026-03-03','Delivered',FALSE,0,1.8,82.00,'Retail','Southwest','Q1-2026'),
('SHIP-010','FedEx','Minneapolis','MN','Milwaukee','WI','Standard Ground','2026-03-08','2026-03-12','Delivered',FALSE,0,7.2,29.00,'SMB','Midwest','Q1-2026'),
('SHIP-011','FedEx','Boston','MA','Providence','RI','Express','2026-04-05','2026-04-07','Delivered',FALSE,0,3.0,47.00,'Enterprise','Northeast','Q2-2026'),
('SHIP-012','FedEx','Nashville','TN','Louisville','KY','Standard Ground','2026-04-12','2026-04-16','Delivered',FALSE,0,11.2,31.00,'Retail','Southeast','Q2-2026'),
('SHIP-013','FedEx','San Francisco','CA','Sacramento','CA','Overnight','2026-04-20','2026-04-21','Delivered',FALSE,0,2.5,86.00,'Healthcare','West','Q2-2026'),
('SHIP-014','FedEx','Philadelphia','PA','Baltimore','MD','Standard Ground','2026-02-14','2026-02-18','Delivered',TRUE,2,6.5,30.00,'SMB','Northeast','Q1-2026'),
('SHIP-015','FedEx','Kansas City','MO','Omaha','NE','Express','2026-05-02','2026-05-05','Delivered',TRUE,1,4.8,54.00,'Retail','Midwest','Q2-2026'),
('SHIP-016','UPS','New York','NY','Newark','NJ','Standard Ground','2026-01-06','2026-01-10','Delivered',FALSE,0,9.1,27.00,'Retail','Northeast','Q1-2026'),
('SHIP-017','UPS','Chicago','IL','Indianapolis','IN','Express','2026-01-13','2026-01-15','Delivered',FALSE,0,5.6,52.00,'Enterprise','Midwest','Q1-2026'),
('SHIP-018','UPS','Dallas','TX','San Antonio','TX','Standard Ground','2026-01-22','2026-01-26','Delivered',FALSE,0,13.3,31.00,'SMB','Southwest','Q1-2026'),
('SHIP-019','UPS','Los Angeles','CA','Las Vegas','NV','Overnight','2026-02-01','2026-02-02','Delivered',FALSE,0,2.2,88.00,'Healthcare','West','Q1-2026'),
('SHIP-020','UPS','Miami','FL','Fort Lauderdale','FL','Standard Ground','2026-02-08','2026-02-12','Delivered',FALSE,0,7.4,26.00,'Retail','Southeast','Q1-2026'),
('SHIP-021','UPS','Seattle','WA','Spokane','WA','Express','2026-02-17','2026-02-19','Delivered',FALSE,0,4.9,50.00,'Enterprise','West','Q1-2026'),
('SHIP-022','UPS','Atlanta','GA','Savannah','GA','Standard Ground','2026-03-05','2026-03-09','Delivered',FALSE,0,10.0,28.00,'SMB','Southeast','Q1-2026'),
('SHIP-023','UPS','Denver','CO','Albuquerque','NM','Express','2026-03-14','2026-03-16','Delivered',FALSE,0,3.7,49.00,'Healthcare','Southwest','Q1-2026'),
('SHIP-024','UPS','Boston','MA','Hartford','CT','Standard Ground','2026-04-01','2026-04-05','Delivered',FALSE,0,8.3,27.00,'Retail','Northeast','Q2-2026'),
('SHIP-025','UPS','Houston','TX','Baton Rouge','LA','Standard Ground','2026-04-10','2026-04-14','Delivered',FALSE,0,15.1,33.00,'Enterprise','Southwest','Q2-2026'),
('SHIP-026','UPS','Chicago','IL','St. Louis','MO','Express','2026-04-18','2026-04-20','Delivered',FALSE,0,6.2,51.00,'SMB','Midwest','Q2-2026'),
('SHIP-027','UPS','Portland','OR','Eugene','OR','Overnight','2026-05-05','2026-05-06','Delivered',FALSE,0,1.9,84.00,'Retail','West','Q2-2026'),
('SHIP-028','UPS','New York','NY','Philadelphia','PA','Standard Ground','2026-01-28','2026-01-31','Delivered',TRUE,1,5.5,27.00,'SMB','Northeast','Q1-2026'),
('SHIP-029','UPS','Memphis','TN','Nashville','TN','Express','2026-03-20','2026-03-23','Delivered',TRUE,1,4.0,52.00,'Retail','Southeast','Q1-2026'),
('SHIP-030','UPS','Minneapolis','MN','Green Bay','WI','Standard Ground','2026-05-12','2026-05-17','Delayed',TRUE,3,11.8,31.00,'Enterprise','Midwest','Q2-2026'),
('SHIP-031','DHL','New York','NY','Boston','MA','Express','2026-01-08','2026-01-10','Delivered',FALSE,0,3.2,52.00,'Enterprise','Northeast','Q1-2026'),
('SHIP-032','DHL','Miami','FL','Orlando','FL','Overnight','2026-01-22','2026-01-23','Delivered',FALSE,0,2.8,81.00,'Healthcare','Southeast','Q1-2026'),
('SHIP-033','DHL','Chicago','IL','Indianapolis','IN','Standard Ground','2026-02-03','2026-02-07','Delivered',FALSE,0,14.0,31.00,'SMB','Midwest','Q1-2026'),
('SHIP-034','DHL','Austin','TX','San Antonio','TX','Standard Ground','2026-02-25','2026-02-28','Delivered',FALSE,0,9.5,29.00,'Retail','Southwest','Q1-2026'),
('SHIP-035','DHL','San Francisco','CA','Sacramento','CA','Express','2026-03-05','2026-03-07','Delivered',FALSE,0,5.1,47.00,'Enterprise','West','Q1-2026'),
('SHIP-036','DHL','Charlotte','NC','Raleigh','NC','Overnight','2026-04-08','2026-04-09','Delivered',FALSE,0,2.3,79.00,'Healthcare','Southeast','Q2-2026'),
('SHIP-037','DHL','Denver','CO','Boulder','CO','Standard Ground','2026-05-01','2026-05-05','Delivered',FALSE,0,7.6,28.00,'SMB','West','Q2-2026'),
('SHIP-038','DHL','Philadelphia','PA','New York','NY','Standard Ground','2026-01-12','2026-01-17','Delivered',TRUE,3,8.2,35.00,'Enterprise','Northeast','Q1-2026'),
('SHIP-039','DHL','Hartford','CT','Providence','RI','Express','2026-01-25','2026-01-29','Delivered',TRUE,2,4.5,55.00,'Retail','Northeast','Q1-2026'),
('SHIP-040','DHL','Detroit','MI','Cleveland','OH','Standard Ground','2026-02-10','2026-02-15','Delivered',TRUE,3,10.2,33.00,'SMB','Midwest','Q1-2026'),
('SHIP-041','DHL','Newark','NJ','Boston','MA','Overnight','2026-02-20','2026-02-23','Delivered',TRUE,1,3.4,90.00,'Healthcare','Northeast','Q1-2026'),
('SHIP-042','DHL','Chicago','IL','Milwaukee','WI','Express','2026-03-10','2026-03-13','Delivered',TRUE,1,5.8,57.00,'Enterprise','Midwest','Q1-2026'),
('SHIP-043','DHL','Pittsburgh','PA','Columbus','OH','Standard Ground','2026-04-02','2026-04-08','Delayed',TRUE,4,12.7,36.00,'Retail','Northeast','Q2-2026'),
('SHIP-044','DHL','Baltimore','MD','Washington','DC','Express','2026-04-15','2026-04-19','Delivered',TRUE,2,4.1,58.00,'SMB','Northeast','Q2-2026'),
('SHIP-045','DHL','Cleveland','OH','Cincinnati','OH','Standard Ground','2026-05-08','2026-05-13','Delayed',TRUE,3,9.3,34.00,'Enterprise','Midwest','Q2-2026'),
('SHIP-046','USPS','New York','NY','Brooklyn','NY','Standard Ground','2026-01-07','2026-01-11','Delivered',FALSE,0,2.5,18.00,'Retail','Northeast','Q1-2026'),
('SHIP-047','USPS','Chicago','IL','Evanston','IL','Standard Ground','2026-01-14','2026-01-18','Delivered',FALSE,0,3.1,16.00,'SMB','Midwest','Q1-2026'),
('SHIP-048','USPS','Los Angeles','CA','Long Beach','CA','Standard Ground','2026-01-21','2026-01-25','Delivered',FALSE,0,4.2,20.00,'Retail','West','Q1-2026'),
('SHIP-049','USPS','Houston','TX','Pasadena','TX','Standard Ground','2026-02-04','2026-02-08','Delivered',FALSE,0,5.0,19.00,'SMB','Southwest','Q1-2026'),
('SHIP-050','USPS','Atlanta','GA','Marietta','GA','Standard Ground','2026-02-11','2026-02-15','Delivered',FALSE,0,2.8,17.00,'Retail','Southeast','Q1-2026'),
('SHIP-051','USPS','Philadelphia','PA','Trenton','NJ','Standard Ground','2026-02-23','2026-02-27','Delivered',FALSE,0,3.5,18.00,'SMB','Northeast','Q1-2026'),
('SHIP-052','USPS','Seattle','WA','Tacoma','WA','Standard Ground','2026-03-03','2026-03-07','Delivered',FALSE,0,4.8,19.00,'Retail','West','Q1-2026'),
('SHIP-053','USPS','Denver','CO','Aurora','CO','Standard Ground','2026-03-17','2026-03-21','Delivered',FALSE,0,3.9,18.00,'SMB','West','Q1-2026'),
('SHIP-054','USPS','Miami','FL','Hialeah','FL','Standard Ground','2026-04-06','2026-04-10','Delivered',FALSE,0,2.2,17.00,'Retail','Southeast','Q2-2026'),
('SHIP-055','USPS','Dallas','TX','Fort Worth','TX','Standard Ground','2026-04-22','2026-04-26','Delivered',FALSE,0,5.6,20.00,'SMB','Southwest','Q2-2026'),
('SHIP-056','USPS','Boston','MA','Cambridge','MA','Standard Ground','2026-05-03','2026-05-07','Delivered',FALSE,0,1.9,16.00,'Retail','Northeast','Q2-2026'),
('SHIP-057','USPS','New York','NY','Yonkers','NY','Standard Ground','2026-01-30','2026-02-04','Delivered',TRUE,3,3.0,18.00,'Retail','Northeast','Q1-2026'),
('SHIP-058','USPS','Chicago','IL','Cicero','IL','Standard Ground','2026-03-25','2026-03-29','Delivered',TRUE,2,2.7,17.00,'SMB','Midwest','Q1-2026'),
('SHIP-059','USPS','Los Angeles','CA','Pasadena','CA','Standard Ground','2026-04-28','2026-05-03','Delayed',TRUE,3,4.1,19.00,'Retail','West','Q2-2026'),
('SHIP-060','USPS','Houston','TX','Sugar Land','TX','Standard Ground','2026-05-15','2026-05-20','Delayed',TRUE,3,3.3,18.00,'SMB','Southwest','Q2-2026');

SELECT COUNT(*) AS SHIPMENTS_LOADED FROM LOGISTICS_HOL.ANALYTICS.SHIPMENTS;

In [ ]:
%%sql -r LoadInc1
INSERT INTO LOGISTICS_HOL.ANALYTICS.DELIVERY_INCIDENTS
  (INCIDENT_ID, SHIPMENT_ID, CARRIER, INCIDENT_DATE, INCIDENT_TYPE, REGION, REPORT_TEXT)
VALUES
('INC-001','SHIP-038','DHL','2026-01-14','Mechanical Failure','Northeast','Driver reported vehicle breakdown on I-95 near Baltimore. The DHL delivery truck experienced a transmission failure mid-route. Package transferred to a secondary vehicle but a 3-hour delay was incurred during the handoff. DHL dispatch was unable to reroute in time to meet the scheduled delivery window. Customer (Enterprise segment) contacted support to escalate. This is the second mechanical issue on this lane in January.'),
('INC-002','SHIP-039','DHL','2026-01-26','Facility Congestion','Northeast','Newark distribution hub experienced severe dock congestion due to a staffing shortage over the weekend. Multiple DHL packages held at the facility for 48 hours. The dock supervisor noted that two loading bays were closed due to equipment maintenance and the remaining bays were overwhelmed. Customer called to report the delay and flagged it was the third late DHL delivery in two months on the Hartford-Providence corridor.'),
('INC-003','SHIP-040','DHL','2026-02-11','Mechanical Failure','Midwest','DHL vehicle experienced brake system warning light activation on I-90 between Detroit and Cleveland. Driver was required by company protocol to pull off at the nearest weigh station and await a maintenance inspection before continuing. Inspection took 4 hours. Package arrived 3 days late. Fleet manager noted that the vehicle had been flagged for brake service two weeks prior but had not yet been serviced due to maintenance backlog.'),
('INC-004','SHIP-041','DHL','2026-02-21','Driver Error','Northeast','Package destined for Boston was mistakenly loaded onto a Newark-bound truck at the Secaucus sorting facility. Error was discovered during the evening scan at the Newark depot. Package was re-routed the following morning, resulting in a 1-day delay to an overnight shipment. The Healthcare customer had scheduled a medical device delivery around the original overnight window. Customer relations was notified and a service credit was issued.'),
('INC-005','SHIP-042','DHL','2026-03-11','Facility Congestion','Midwest','DHL Chicago hub experienced a 6-hour processing delay due to a conveyor belt malfunction in the primary sorting system. Approximately 200 packages were held and processed manually, significantly slowing throughput. The Milwaukee-bound batch including this shipment was held overnight rather than dispatched on the scheduled truck. Customer escalated via chat after tracking showed no movement for 18 hours.'),
('INC-006','SHIP-043','DHL','2026-04-03','Mechanical Failure','Northeast','Vehicle breakdown on the Pennsylvania Turnpike near Breezewood. DHL driver reported the check engine light had been on for three days prior to the route but the vehicle was dispatched without inspection. The engine overheated and required towing. A replacement vehicle was dispatched from Pittsburgh but arrived 6 hours later. The package experienced a 4-day cumulative delay. Customer (Retail) contacted support twice during the incident window and requested escalation to a supervisor.'),
('INC-007','SHIP-044','DHL','2026-04-16','Facility Congestion','Northeast','Baltimore sorting facility experienced a 2-day processing backlog due to a combination of increased volume from a major retail promotion and reduced staffing over the Easter holiday weekend. DHL failed to add supplementary capacity despite advance notice of the volume spike. Multiple Express shipments, including this one, were downgraded to Standard Ground processing. Customer was not proactively notified of the change in service level.'),
('INC-008','SHIP-045','DHL','2026-05-09','Mechanical Failure','Midwest','Third DHL mechanical failure on the Cleveland-Cincinnati route this quarter. Driver reported steering column vibration that worsened over the 250-mile route, eventually requiring a roadside stop near Columbus. DHL roadside assistance took 2.5 hours to arrive. Package was transferred to a courier vehicle for the final 80 miles. Customer stated: This is the fourth late DHL delivery we have received in Q2. We are evaluating switching to an alternative carrier for this lane.'),
('INC-009','SHIP-028','UPS','2026-01-29','Weather Delay','Northeast','Winter storm Juno brought 8 inches of snow to the New York metro area on January 28-29, causing road closures on several key arterial routes in New Jersey. UPS driver attempted delivery but the customer address was inaccessible due to an unplowed private road. Package was held at the local depot overnight and redelivered the following morning after roads were cleared. No service failure on UPS part, force majeure delay.'),
('INC-010','SHIP-029','UPS','2026-03-21','Damaged Package','Southeast','Package arrived at the Memphis UPS facility showing external crush damage on the lower corner. Internal contents (industrial measuring equipment) were intact but the outer casing showed a hairline crack. Customer requested a replacement unit. UPS claims department opened a formal investigation. Origin facility review showed the package was loaded under a heavier freight pallet without adequate cushioning.');

SELECT COUNT(*) AS LOADED_SO_FAR FROM LOGISTICS_HOL.ANALYTICS.DELIVERY_INCIDENTS;

In [ ]:
%%sql -r LoadInc3
INSERT INTO LOGISTICS_HOL.ANALYTICS.DELIVERY_INCIDENTS
  (INCIDENT_ID, SHIPMENT_ID, CARRIER, INCIDENT_DATE, INCIDENT_TYPE, REGION, REPORT_TEXT)
VALUES
('INC-021','SHIP-039','DHL','2026-01-27','Facility Congestion','Northeast','Newark facility congestion affected 45 DHL packages scheduled for the Hartford-Providence delivery run. The receiving supervisor noted the root cause was a lack of weekend supervisor coverage combined with two dock doors out of service for maintenance. DHL regional operations confirmed the facility had flagged the dock maintenance issue two weeks earlier but repair was deprioritized. The recurring congestion at Newark has been flagged in the Q1 operations review as a systemic risk requiring capacity planning intervention.'),
('INC-022','SHIP-042','DHL','2026-03-12','Customer Complaint','Midwest','Customer service log: Retail customer contacted DHL support after tracking showed no movement for 18 hours on an Express shipment. Support confirmed package was held at Chicago hub due to conveyor malfunction. Customer stated: This is the second DHL Express delay this quarter. We will be reviewing our DHL account at our next logistics review meeting in April.'),
('INC-023','SHIP-044','DHL','2026-04-17','Customer Complaint','Northeast','SMB customer escalated following service level downgrade from Express to Standard Ground without notification. Customer had chosen Express shipping for a product launch deadline. Customer escalation email: We paid for Express and received Standard Ground service. DHL did not notify us of the service change. Our product launch event was delayed by 24 hours as a result. This is unacceptable and we request a full refund plus compensation for the event rescheduling cost.'),
('INC-024','SHIP-046','USPS','2026-01-12','Weather Delay','Northeast','Light snow accumulation in Brooklyn on January 11 caused minor delays on pedestrian delivery routes. Carrier route 09-067 was partially suspended for 4 hours pending sidewalk clearing. Package delivered one day late. Customer left a voicemail noting the delay but no formal complaint filed.'),
('INC-025','SHIP-059','USPS','2026-04-29','Facility Congestion','West','Los Angeles USPS processing center experienced elevated volume following a 3-day weekend. Standard Ground packages received on April 27-28 were not processed until April 30, resulting in a 2-day delay in dispatch. Customer tracking showed no movement for 48 hours. The Los Angeles facility has experienced similar post-weekend processing delays in each of the past three months.'),
('INC-026','SHIP-060','USPS','2026-05-16','Weather Delay','Southwest','Heavy rainfall and localized flooding in the Greater Houston area on May 15-16 disrupted delivery operations for multiple USPS routes in Sugar Land and Missouri City. Three carrier routes were suspended for 24 hours. Package arrived 3 days late due to flooding delay followed by a secondary processing backlog at the Stafford USPS distribution center. Standard weather delay event.'),
('INC-027','SHIP-038','DHL','2026-01-15','Mechanical Failure','Northeast','Fleet maintenance review triggered by the January 14 breakdown revealed that 3 of the 7 DHL vehicles assigned to the Philadelphia-New York corridor had overdue service intervals exceeding 30 days. The regional fleet manager confirmed that budget constraints during Q4 2025 had delayed preventive maintenance across the Northeast fleet. This systemic maintenance backlog is the root cause of the recent mechanical failure pattern and has been escalated to the DHL Northeast operations director for immediate remediation.'),
('INC-028','SHIP-043','DHL','2026-04-05','Mechanical Failure','Northeast','Post-incident review of the Pennsylvania Turnpike breakdown confirmed the vehicle had check engine codes for coolant temperature sensor malfunction logged for 72 hours prior to dispatch. DHL pre-trip inspection protocol requires any active engine codes to be cleared or investigated before a vehicle is dispatched on a long-haul route. The dispatch supervisor stated the vehicle was released due to a shortage of available vehicles that day. This represents a policy compliance failure and has been referred to DHL safety and compliance for review.'),
('INC-029','SHIP-031','DHL','2026-01-09','Customer Complaint','Northeast','Enterprise customer provided mixed feedback following on-time delivery on the New York-Boston Express route. Customer noted: DHL Express performance on this lane has been reliable this month. However, we have started to notice an uptick in delays on our Northeast Standard Ground shipments, particularly out of the Philadelphia and Newark hubs. We plan to raise this in our quarterly carrier review and would like DHL to provide a performance summary for the Northeast region before our March meeting.'),
('INC-030','SHIP-016','UPS','2026-01-10','Weather Delay','Northeast','Moderate snowstorm on January 9-10 slowed UPS ground operations across the New York metro area. Standard Ground deliveries from the Kearny NJ hub experienced 6-8 hour delays. Package delivered within business hours on the scheduled delivery date despite adverse conditions. No complaint filed. UPS operations manager confirmed the route was completed without service failure classification due to the weather exemption policy.');

SELECT COUNT(*) AS TOTAL_INCIDENTS FROM LOGISTICS_HOL.ANALYTICS.DELIVERY_INCIDENTS;

In [ ]:
%%sql -r RowCounts
SELECT 'SHIPMENTS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM LOGISTICS_HOL.ANALYTICS.SHIPMENTS
UNION ALL SELECT 'DELIVERY_INCIDENTS', COUNT(*) FROM LOGISTICS_HOL.ANALYTICS.DELIVERY_INCIDENTS
ORDER BY TABLE_NAME;

In [ ]:
%%sql -r CarrierSummary
-- On-time rate by carrier (what Cortex Analyst will answer via natural language)
SELECT CARRIER, COUNT(*) AS TOTAL,
  SUM(CASE WHEN NOT IS_LATE THEN 1 ELSE 0 END) AS ON_TIME,
  SUM(CASE WHEN IS_LATE THEN 1 ELSE 0 END) AS LATE,
  ROUND(SUM(CASE WHEN NOT IS_LATE THEN 1.0 ELSE 0.0 END)/COUNT(*)*100,1) AS ON_TIME_RATE_PCT
FROM LOGISTICS_HOL.ANALYTICS.SHIPMENTS
GROUP BY CARRIER ORDER BY ON_TIME_RATE_PCT DESC;

In [ ]:
%%sql -r SampleIncidents
SELECT INCIDENT_ID, CARRIER, INCIDENT_TYPE, REGION, LEFT(REPORT_TEXT,150) AS REPORT_PREVIEW
FROM LOGISTICS_HOL.ANALYTICS.DELIVERY_INCIDENTS ORDER BY CARRIER, INCIDENT_DATE LIMIT 8;